# Learn RAG Series using vector database chromadb

## Import Libraries

In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import NLTKTextSplitter
from langchain_chroma import Chroma
from IPython.display import Markdown as md

/home/kevadamar/Projects/Research/ragl/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/kevadamar/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Config Process

In [3]:
!curl -o ai_paper.pdf https://www.pearsonvue.com/content/dam/VUE/vue/en/documents/clients/it-specialist/its-od-307-artificial-intel-pearson.pdf

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  259k  100  259k    0     0   874k      0 --:--:-- --:--:-- --:--:--  876k


In [4]:
from langchain_community.document_loaders import PyPDFLoader
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

# load pdf data
loader = PyPDFLoader("ai_paper.pdf")
pages = loader.load_and_split()
print(f"Loaded {len(pages)} pages from the PDF document.")

# Initialize the chat model
chat_model = ChatGoogleGenerativeAI(
    google_api_key=GEMINI_API_KEY, 
    model="gemini-2.5-flash",
    temperature=0.9
)

Loaded 5 pages from the PDF document.


## Chunking Data

In [5]:
text_splitter = NLTKTextSplitter(
    chunk_size=500, 
    chunk_overlap=50
)

chunks = text_splitter.split_documents(pages)
print(f"Split into {len(chunks)} chunks.")

Created a chunk of size 579, which is longer than the specified 500
Created a chunk of size 1239, which is longer than the specified 500
Created a chunk of size 1277, which is longer than the specified 500


Split into 14 chunks.


## Embedding model

In [6]:
embedding_model = GoogleGenerativeAIEmbeddings(
    google_api_key=GEMINI_API_KEY,
    model="gemini-embedding-001"
)
embedding_model

GoogleGenerativeAIEmbeddings(client=<google.genai.client.Client object at 0x7f26787b95b0>, model='gemini-embedding-001', task_type=None, google_api_key=SecretStr('**********'), credentials=None, vertexai=None, project=None, location=None, base_url=None, additional_headers=None, client_args=None, request_options=None, output_dimensionality=None)

## Vector Databases

In [7]:
persist_directory="./chroma_db"
# Create directory if it doesn't exist
os.makedirs(persist_directory, exist_ok=True)

db = Chroma.from_documents(
    chunks,
    embedding_model,
    persist_directory=persist_directory
)

In [8]:
# Connection db
db_connection = Chroma(
    persist_directory=persist_directory, 
    embedding_function=embedding_model
)

retriever=db_connection.as_retriever(search_kwargs={"k": 12})

In [9]:
# Check response
query = "Jelaskan apa saja skill yang di butuhkan untuk sertifikasi AI di person vue?"
query2 = "Apa saja topik yang di bahas dalam sertifikasi AI di pearson vue?"
query3 = "Apa saja skill untuk bisa menjadi AI engineer?"
response = retriever.invoke(query)
md(response[0].page_content)

Candidates at least 150 hours of 
instruction and/or exploration of artificial intelligence methodology and solutions.

## Prompting LLM

In [10]:
# promp template
from langchain_core.messages import SystemMessage
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate

system_text = """Anda adalah asisten AI yang sangat pintar dan membantu pengguna dengan menjawab pertanyaan mereka berdasarkan konteks yang diberikan."""
human_text = """
Jawab pertanyaan berikut berdasarkan konteks yang diberikan.

Konteks: {context}
Pertanyaan: {question}

Provide a detailed answer.
Don’t justify your answers.
Don’t give information not mentioned in the CONTEXT INFORMATION.
Do not say "according to the context" or "mentioned in the context" or similar.
Do not use words like "As an AI language model".
If the answer is not contained within the text below, say "I don't know"."""

chat_template = ChatPromptTemplate.from_messages([
    SystemMessage(content=system_text),
    HumanMessagePromptTemplate.from_template(human_text)
])


In [11]:
# parser output
from langchain_core.output_parsers import StrOutputParser
output_parser = StrOutputParser()

def format_docs(docs):
    formatted = ""
    for i, doc in enumerate(docs):
        formatted += f"Source {i+1}:\n{doc.page_content}\n\n"
    return formatted.strip()

In [12]:
# test formatting
formatted_docs = format_docs(response)
md(formatted_docs)

Source 1:
Candidates at least 150 hours of 
instruction and/or exploration of artificial intelligence methodology and solutions.

Source 2:
Candidates at least 150 hours of 
instruction and/or exploration of artificial intelligence methodology and solutions.

Source 3:
Successful candidates will be able to analyze and classify a problem.

They should 
be able to demonstrate knowledge of data collection, data processing, and feature engineering strategies.

Candidates should be able to choose an appropriate algorithm for training a model, and understand the 
metrics used to evaluate model performance.

They should understand the AI development lifecycle and 
how a production pipeline is used to allow for continuous improvement.

Source 4:
Successful candidates will be able to analyze and classify a problem.

They should 
be able to demonstrate knowledge of data collection, data processing, and feature engineering strategies.

Candidates should be able to choose an appropriate algorithm for training a model, and understand the 
metrics used to evaluate model performance.

They should understand the AI development lifecycle and 
how a production pipeline is used to allow for continuous improvement.

Source 5:
and the problem
• Determine problem type (e.g., classification, regression, unsupervised, 
reinforcement)
1.3 Identify the areas of expertise needed to solve the problem
• Identify business expertise required
• Identify the need for domain (subject-matter) expertise on the problem
• Identify AI expertise needed
• Identify implementation expertise needed
1.4 Build a security plan
• Consider internal access levels or permissions
• Consider infrastructure security
• Assess the risk of using a certain model or potential attack surfaces (e.g., 
adversarial attacks on real-time learning model)
1.5 Ensure that AI is used appropriately
• Identify potential ways that the AI can mispredict or harm specific user 
groups
• Set guidelines for data gathering and use
• Set guidelines for algorithm selection from user perspective
• Consider how the subject of the data can interpret the results
• Consider out-of-context use of AI results
IT SPECIALIST EXAM OBJECTIVES
Artificial Intelligence
Candidates for this certification have a foundational knowledge the procedures used to develop an artificial 
intelligence (AI) solution, as well as an understanding of the issues surrounding the governance, transparency, 
security, and ethics of AI.

Source 6:
and the problem
• Determine problem type (e.g., classification, regression, unsupervised, 
reinforcement)
1.3 Identify the areas of expertise needed to solve the problem
• Identify business expertise required
• Identify the need for domain (subject-matter) expertise on the problem
• Identify AI expertise needed
• Identify implementation expertise needed
1.4 Build a security plan
• Consider internal access levels or permissions
• Consider infrastructure security
• Assess the risk of using a certain model or potential attack surfaces (e.g., 
adversarial attacks on real-time learning model)
1.5 Ensure that AI is used appropriately
• Identify potential ways that the AI can mispredict or harm specific user 
groups
• Set guidelines for data gathering and use
• Set guidelines for algorithm selection from user perspective
• Consider how the subject of the data can interpret the results
• Consider out-of-context use of AI results
IT SPECIALIST EXAM OBJECTIVES
Artificial Intelligence
Candidates for this certification have a foundational knowledge the procedures used to develop an artificial 
intelligence (AI) solution, as well as an understanding of the issues surrounding the governance, transparency, 
security, and ethics of AI.

Source 7:
Data Collection, Processing, and Engineering 
2.1 Choose the way to collect data
• Determine type/characteristics of data needed
• Decide if there is an existing dataset or if you need to generate your own
• When generating your own dataset, decide whether collection can be 
automated or requires user input
2.2 Assess data quality     
• Determine whether the dataset meets needs of task
• Look for missing or corrupt data elements
2.3 Ensure that data are representative  
• Examine collection techniques for potential sources of bias
• Make sure the amount of data is enough to build an unbiased model
2.4 Identify resource requirements (e.g., computing, time complexity)
• Assess whether the problem is solvable with available computing resources
• Consider the budget of the project and the resources that are available
2.5 Convert data into suitable formats (e.g., numerical, image, time 
series)
• Convert data to binary (e.g., images become pixels)
• Convert computer data into features suitable for AI (e.g., sentences 
become tokens)
2.6 Select features for the AI model 
• Determine which data features to include 
• Build initial feature vectors for test/train dataset
• Consult with subject-matter experts to confirm feature selection
2.7 Engage in feature engineering
• Review features and determine what standard transformations are needed
• Create processed datasets
2.8 Identify training and test datasets 
• Separate available data into training and test datasets
• Ensure test dataset is represented 
2.9 Document data decisions   
• List assumptions, predicates, and constraints upon which design choices 
have been reasoned
• Make this information available to regulators and end users who demand 
deep transparency 
IT SPECIALIST EXAM OBJECTIVES

Source 8:
Data Collection, Processing, and Engineering 
2.1 Choose the way to collect data
• Determine type/characteristics of data needed
• Decide if there is an existing dataset or if you need to generate your own
• When generating your own dataset, decide whether collection can be 
automated or requires user input
2.2 Assess data quality     
• Determine whether the dataset meets needs of task
• Look for missing or corrupt data elements
2.3 Ensure that data are representative  
• Examine collection techniques for potential sources of bias
• Make sure the amount of data is enough to build an unbiased model
2.4 Identify resource requirements (e.g., computing, time complexity)
• Assess whether the problem is solvable with available computing resources
• Consider the budget of the project and the resources that are available
2.5 Convert data into suitable formats (e.g., numerical, image, time 
series)
• Convert data to binary (e.g., images become pixels)
• Convert computer data into features suitable for AI (e.g., sentences 
become tokens)
2.6 Select features for the AI model 
• Determine which data features to include 
• Build initial feature vectors for test/train dataset
• Consult with subject-matter experts to confirm feature selection
2.7 Engage in feature engineering
• Review features and determine what standard transformations are needed
• Create processed datasets
2.8 Identify training and test datasets 
• Separate available data into training and test datasets
• Ensure test dataset is represented 
2.9 Document data decisions   
• List assumptions, predicates, and constraints upon which design choices 
have been reasoned
• Make this information available to regulators and end users who demand 
deep transparency 
IT SPECIALIST EXAM OBJECTIVES

Source 9:
Copyright © 2025 Pearson Education, Inc. or its affiliates(s).

All rights reserved.

IT SPECIALIST EXAM OBJECTIVES
5.3 Measure impacts on individuals and communities
• Analyze impact on specific subgroups
• Identify and mitigate issues
• Identify opportunities for optimization
5.4 Handle feedback from users
• Measure user satisfaction
• Assess whether users are confused (e.g., do they understand what the AI is 
supposed to do for them?)

Source 10:
Copyright © 2025 Pearson Education, Inc. or its affiliates(s).

All rights reserved.

IT SPECIALIST EXAM OBJECTIVES
5.3 Measure impacts on individuals and communities
• Analyze impact on specific subgroups
• Identify and mitigate issues
• Identify opportunities for optimization
5.4 Handle feedback from users
• Measure user satisfaction
• Assess whether users are confused (e.g., do they understand what the AI is 
supposed to do for them?)

Source 11:
Copyright © 2025 Pearson Education, Inc. or its affiliates(s).

All rights reserved.

IT SPECIALIST EXAM OBJECTIVES
3.

AI Algorithms and Models  
3.1 Consider applicability of specific algorithms 
• Evaluate AI algorithm families
• Decide which algorithms  are suitable, e.g., neural network, classification 
(like decision tree, k means)
3.2 Train a model using the selected algorithm 
• Train model for an algorithm with best-guess starting parameters.

Source 12:
Copyright © 2025 Pearson Education, Inc. or its affiliates(s).

All rights reserved.

IT SPECIALIST EXAM OBJECTIVES
3.

AI Algorithms and Models  
3.1 Consider applicability of specific algorithms 
• Evaluate AI algorithm families
• Decide which algorithms  are suitable, e.g., neural network, classification 
(like decision tree, k means)
3.2 Train a model using the selected algorithm 
• Train model for an algorithm with best-guess starting parameters.

In [13]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | chat_template
    | chat_model
    | output_parser
)

final_response = rag_chain.invoke(query)

print(f"the question: {query}\n")
md(final_response)

the question: Jelaskan apa saja skill yang di butuhkan untuk sertifikasi AI di person vue?



Skill yang dibutuhkan untuk sertifikasi AI meliputi:

*   **Pemahaman Dasar dan Prosedur Pengembangan AI:** Memiliki pengetahuan dasar tentang prosedur yang digunakan untuk mengembangkan solusi AI, serta memahami isu-isu seputar tata kelola, transparansi, keamanan, dan etika AI.
*   **Analisis dan Klasifikasi Masalah:** Mampu menganalisis dan mengklasifikasikan masalah, termasuk menentukan jenis masalah (misalnya, klasifikasi, regresi, tanpa pengawasan, penguatan).
*   **Pengetahuan tentang Pengumpulan, Pemrosesan, dan Rekayasa Fitur Data:**
    *   Mampu memilih cara mengumpulkan data, termasuk menentukan jenis/karakteristik data yang dibutuhkan, memutuskan apakah ada kumpulan data yang ada atau perlu dibuat sendiri, dan apakah pengumpulan dapat diotomatisasi atau memerlukan masukan pengguna.
    *   Mampu menilai kualitas data dan memastikan data representatif.
    *   Mampu mengidentifikasi persyaratan sumber daya (misalnya, komputasi, kompleksitas waktu).
    *   Mampu mengonversi data ke format yang sesuai (misalnya, numerik, gambar, deret waktu, biner, atau fitur yang cocok untuk AI seperti kalimat menjadi token).
    *   Mampu memilih fitur untuk model AI, membangun vektor fitur awal, dan berkonsultasi dengan ahli materi pelajaran.
    *   Mampu melakukan rekayasa fitur, meninjau fitur, menentukan transformasi standar yang diperlukan, dan membuat kumpulan data yang diproses.
    *   Mampu mengidentifikasi kumpulan data pelatihan dan pengujian, serta memastikan kumpulan data pengujian terwakili.
    *   Mampu mendokumentasikan keputusan data.
*   **Pemilihan dan Pelatihan Algoritma AI:**
    *   Mampu memilih algoritma yang sesuai untuk melatih model.
    *   Memahami metrik yang digunakan untuk mengevaluasi kinerja model.
    *   Mampu mempertimbangkan penerapan algoritma tertentu, mengevaluasi keluarga algoritma AI, dan memutuskan algoritma mana yang cocok (misalnya, jaringan saraf, klasifikasi seperti pohon keputusan, k-means).
    *   Mampu melatih model menggunakan algoritma yang dipilih dengan parameter awal terbaik.
*   **Pemahaman Siklus Hidup Pengembangan AI dan Pipeline Produksi:** Memahami siklus hidup pengembangan AI dan bagaimana pipeline produksi digunakan untuk memungkinkan peningkatan berkelanjutan.
*   **Identifikasi Keahlian yang Dibutuhkan:** Mampu mengidentifikasi keahlian bisnis, domain (materi pelajaran), AI, dan implementasi yang diperlukan untuk memecahkan masalah.
*   **Perencanaan Keamanan:** Mampu membangun rencana keamanan, termasuk mempertimbangkan tingkat akses internal atau izin, keamanan infrastruktur, dan menilai risiko penggunaan model tertentu atau permukaan serangan potensial.
*   **Penggunaan AI yang Tepat:** Mampu memastikan AI digunakan dengan tepat, termasuk mengidentifikasi cara potensial AI dapat salah memprediksi atau membahayakan kelompok pengguna tertentu, menetapkan pedoman pengumpulan dan penggunaan data, menetapkan pedoman pemilihan algoritma dari perspektif pengguna, mempertimbangkan bagaimana subjek data dapat menafsirkan hasil, dan mempertimbangkan penggunaan hasil AI di luar konteks.
*   **Pengukuran Dampak dan Penanganan Umpan Balik:**
    *   Mampu mengukur dampak pada individu dan komunitas, menganalisis dampak pada subkelompok tertentu, mengidentifikasi dan mengurangi masalah, serta mengidentifikasi peluang untuk optimasi.
    *   Mampu menangani umpan balik dari pengguna, mengukur kepuasan pengguna, dan menilai apakah pengguna bingung.

In [16]:
# test query 2
print(f"the question: {query2}\n")
md(rag_chain.invoke(query2))

the question: Apa saja topik yang di bahas dalam sertifikasi AI di pearson vue?



Topik yang dibahas dalam sertifikasi AI di Pearson Vue meliputi:

*   Pengetahuan dasar tentang prosedur yang digunakan untuk mengembangkan solusi kecerdasan buatan (AI).
*   Pemahaman tentang masalah seputar tata kelola, transparansi, keamanan, dan etika AI.
*   Menganalisis dan mengklasifikasikan masalah.
*   Mengidentifikasi jenis masalah (misalnya, klasifikasi, regresi, tanpa pengawasan, *reinforcement*).
*   Mengidentifikasi bidang keahlian yang dibutuhkan untuk memecahkan masalah (keahlian bisnis, keahlian domain, keahlian AI, keahlian implementasi).
*   Membangun rencana keamanan (tingkat akses internal, keamanan infrastruktur, menilai risiko model atau potensi permukaan serangan).
*   Memastikan AI digunakan dengan tepat (mengidentifikasi potensi misprediksi atau bahaya, menetapkan pedoman pengumpulan dan penggunaan data, menetapkan pedoman pemilihan algoritma, mempertimbangkan interpretasi hasil oleh subjek data, mempertimbangkan penggunaan hasil AI di luar konteks).
*   Strategi pengumpulan data, pemrosesan data, dan *feature engineering*.
*   Memilih cara mengumpulkan data (menentukan jenis/karakteristik data, memutuskan dataset yang sudah ada atau membuat sendiri, otomatisasi atau input pengguna).
*   Menilai kualitas data (memenuhi kebutuhan tugas, mencari elemen data yang hilang atau rusak).
*   Memastikan data representatif (memeriksa bias, memastikan jumlah data yang cukup).
*   Mengidentifikasi persyaratan sumber daya (komputasi, kompleksitas waktu, anggaran).
*   Mengubah data ke format yang sesuai (numerik, gambar, deret waktu; biner, token).
*   Memilih fitur untuk model AI (menentukan fitur, membangun vektor fitur awal, berkonsultasi dengan ahli materi pelajaran).
*   Melakukan *feature engineering* (meninjau fitur, transformasi standar, membuat dataset yang diproses).
*   Mengidentifikasi dataset pelatihan dan pengujian (memisahkan data, memastikan dataset pengujian direpresentasikan).
*   Mendokumentasikan keputusan data (daftar asumsi, predikat, batasan).
*   Algoritma dan model AI.
*   Mempertimbangkan penerapan algoritma spesifik (mengevaluasi famili algoritma, memutuskan algoritma yang cocok seperti jaringan saraf, klasifikasi - pohon keputusan, k-means).
*   Melatih model menggunakan algoritma yang dipilih (dengan parameter awal tebakan terbaik).
*   Memilih algoritma yang sesuai untuk melatih model.
*   Memahami metrik yang digunakan untuk mengevaluasi kinerja model.
*   Memahami siklus hidup pengembangan AI dan bagaimana *production pipeline* digunakan untuk perbaikan berkelanjutan.
*   Mengukur dampak pada individu dan komunitas (menganalisis dampak pada subkelompok, mengidentifikasi/mengurangi masalah, mengidentifikasi peluang optimasi).
*   Menangani umpan balik dari pengguna (mengukur kepuasan pengguna, menilai kebingungan pengguna).

In [17]:
# test query 3
print(f"the question: {query3}\n")
md(rag_chain.invoke(query3))

the question: Apa saja skill untuk bisa menjadi AI engineer?



Untuk menjadi AI engineer, kandidat harus memiliki skill-skill berikut:

*   Mampu menganalisis dan mengklasifikasikan masalah.
*   Mendemonstrasikan pengetahuan tentang strategi pengumpulan data, pemrosesan data, dan rekayasa fitur.
*   Mampu memilih algoritma yang tepat untuk melatih model.
*   Memahami metrik yang digunakan untuk mengevaluasi kinerja model.
*   Memahami siklus pengembangan AI dan bagaimana *production pipeline* digunakan untuk memungkinkan peningkatan berkelanjutan.
*   Memiliki setidaknya 150 jam instruksi dan/atau eksplorasi metodologi dan solusi kecerdasan buatan.
*   Memiliki pengetahuan dasar tentang prosedur yang digunakan untuk mengembangkan solusi AI.
*   Memahami masalah seputar tata kelola, transparansi, keamanan, dan etika AI.
*   Mampu menentukan jenis masalah (misalnya, klasifikasi, regresi, tanpa pengawasan, *reinforcement*).
*   Mampu mengidentifikasi keahlian yang dibutuhkan (keahlian bisnis, domain, AI, implementasi).
*   Mampu membangun rencana keamanan (tingkat akses internal, keamanan infrastruktur, menilai risiko model atau *attack surfaces*).
*   Mampu memastikan AI digunakan secara tepat (mengidentifikasi potensi kesalahan prediksi atau kerugian pada kelompok pengguna tertentu, menetapkan pedoman pengumpulan dan penggunaan data, menetapkan pedoman pemilihan algoritma dari perspektif pengguna, mempertimbangkan bagaimana subjek data dapat menginterpretasikan hasil, mempertimbangkan penggunaan hasil AI di luar konteks).

Secara lebih rinci dalam Pengumpulan, Pemrosesan, dan Rekayasa Data:
*   **Memilih cara mengumpulkan data:** Menentukan jenis/karakteristik data yang dibutuhkan, memutuskan apakah ada *dataset* yang sudah ada atau perlu membuatnya sendiri, memutuskan apakah pengumpulan dapat diotomatisasi atau memerlukan masukan pengguna.
*   **Menilai kualitas data:** Menentukan apakah *dataset* memenuhi kebutuhan tugas, mencari elemen data yang hilang atau rusak.
*   **Memastikan data representatif:** Memeriksa teknik pengumpulan untuk potensi sumber bias, memastikan jumlah data cukup untuk membangun model yang tidak bias.
*   **Mengidentifikasi kebutuhan sumber daya:** Menilai apakah masalah dapat dipecahkan dengan sumber daya komputasi yang tersedia, mempertimbangkan anggaran proyek dan sumber daya yang tersedia.
*   **Mengubah data menjadi format yang sesuai:** Mengubah data menjadi biner (misalnya, gambar menjadi piksel), mengubah data komputer menjadi fitur yang cocok untuk AI (misalnya, kalimat menjadi token).
*   **Memilih fitur untuk model AI:** Menentukan fitur data mana yang akan disertakan, membangun vektor fitur awal untuk *dataset* uji/latih, berkonsultasi dengan ahli materi pelajaran untuk mengkonfirmasi pemilihan fitur.
*   **Melakukan rekayasa fitur:** Meninjau fitur dan menentukan transformasi standar apa yang diperlukan, membuat *dataset* yang diproses.
*   **Mengidentifikasi *dataset* pelatihan dan pengujian:** Memisahkan data yang tersedia menjadi *dataset* pelatihan dan pengujian, memastikan *dataset* pengujian terwakili.
*   **Mendokumentasikan keputusan data:** Membuat daftar asumsi, predikat, dan batasan di mana pilihan desain telah dipertimbangkan, membuat informasi ini tersedia bagi regulator dan pengguna akhir yang menuntut transparansi mendalam.

Dalam Definisi Masalah AI:
*   Mengidentifikasi masalah yang ingin dipecahkan menggunakan AI (misalnya, segmentasi pengguna, meningkatkan layanan pelanggan).
*   Mengidentifikasi kebutuhan yang akan ditangani.
*   Mencari tahu informasi yang masuk dan keluaran yang diharapkan.
*   Menentukan apakah AI diperlukan.
*   Mempertimbangkan keuntungan dan kerugian AI dalam situasi tersebut.
*   Menentukan keberhasilan yang terukur.
*   Membadingkan terhadap risiko spesifik domain atau organisasi yang mungkin rentan terhadap proyek tersebut.
*   Mengklasifikasikan masalah (misalnya, regresi, pembelajaran tanpa pengawasan).
*   Memeriksa data yang tersedia (berlabel atau tidak berlabel?).

Dalam Integrasi dan Penerapan Aplikasi:
*   Melatih pelanggan tentang cara menggunakan produk dan apa yang diharapkan darinya (menginformasikan batasan model, penggunaan model yang dimaksudkan, berbagi dokumentasi, mengelola ekspektasi pelanggan).
*   Merencanakan untuk mengatasi potensi tantangan model dalam produksi (memahami jenis tantangan yang mungkin dihadapi, memahami indikator tantangan, memahami bagaimana setiap jenis tantangan dapat dimitigasi).
*   Merancang *production pipeline*, termasuk integrasi aplikasi (membuat *pipeline* untuk pelatihan dan prediksi yang dapat memenuhi kebutuhan produk, menemukan solusi yang berfungsi dengan penyimpanan data yang ada dan terhubung ke aplikasi, membangun koneksi antara AI dan aplikasi, membangun mekanisme untuk mengumpulkan umpan balik pengguna, menguji akurasi AI melalui aplikasi, menguji ketahanan AI, menguji kecepatan AI, menguji aplikasi agar sesuai dengan ukuran kasus penggunaan).
*   Mendukung solusi AI (mendokumentasikan fungsi dalam solusi AI untuk memungkinkan pemeliharaan, melatih tim dukungan, mengimplementasikan mekanisme umpan balik, mengimplementasikan detektor *drift*, mengimplementasikan cara untuk mengumpulkan data baru).